# Fleet MOT Failure Risk Model

## Notebook 02: Data Cleaning and Preparation

This notebook focuses on cleaning and preparing the 2024 MOT sample dataset for analysis.

The purpose of this notebook is to:

- Load the MOT sample dataset
- Standardise column names
- Select useful columns for MOT failure risk analysis
- Convert date columns into datetime format
- Clean MOT result values
- Create pass/fail indicators
- Create vehicle age fields
- Create mileage bands
- Create basic risk features
- Export a cleaned analysis-ready sample file

This notebook prepares the data for exploratory analysis, SQL reporting, Power BI dashboarding and future failure risk modelling.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define project paths

sample_data_path = Path("../data/sample")
processed_data_path = Path("../data/processed")

sample_file = sample_data_path / "mot_2024_sample.csv"
cleaned_output_file = processed_data_path / "mot_2024_cleaned_sample.csv"

print("Sample file exists:", sample_file.exists())
print("Processed data folder exists:", processed_data_path.exists())

Sample file exists: True
Processed data folder exists: True


In [3]:
# Load the MOT sample dataset

mot = pd.read_csv(sample_file, low_memory=False)

print("Dataset loaded successfully.")
print("Rows:", mot.shape[0])
print("Columns:", mot.shape[1])

Dataset loaded successfully.
Rows: 600000
Columns: 16


## Standardise Column Names

The raw MOT dataset may contain column names with spaces, capitals or inconsistent formatting.

To make the data easier to use in Python, SQL and Power BI, I will standardise all column names into lowercase snake_case format.

In [4]:
# Standardise column names

mot_clean = mot.copy()

mot_clean.columns = (
    mot_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

mot_clean.columns.tolist()

['test_id',
 'vehicle_id',
 'test_date',
 'test_class_id',
 'test_type',
 'test_result',
 'test_mileage',
 'postcode_area',
 'make',
 'model',
 'colour',
 'fuel_type',
 'cylinder_capacity',
 'first_use_date',
 'completed_date',
 'source_file']

In [5]:
# Helper function to find likely column names

def find_column(df, possible_names):
    """
    Return the first matching column from a list of possible column names.
    """
    for name in possible_names:
        if name in df.columns:
            return name
    return None


# Identify key columns
test_result_col = find_column(mot_clean, ["test_result", "result"])
test_date_col = find_column(mot_clean, ["test_date", "completed_date"])
first_use_date_col = find_column(mot_clean, ["first_use_date", "date_of_first_use", "first_registration_date"])
mileage_col = find_column(mot_clean, ["test_mileage", "odometer_value", "mileage"])
make_col = find_column(mot_clean, ["make", "vehicle_make"])
model_col = find_column(mot_clean, ["model", "vehicle_model"])
fuel_col = find_column(mot_clean, ["fuel_type", "fuel"])
vehicle_class_col = find_column(mot_clean, ["test_class_id", "vehicle_class", "class"])
colour_col = find_column(mot_clean, ["colour", "color"])
postcode_col = find_column(mot_clean, ["postcode_area", "postcode"])
source_file_col = find_column(mot_clean, ["source_file"])

identified_columns = {
    "test_result_col": test_result_col,
    "test_date_col": test_date_col,
    "first_use_date_col": first_use_date_col,
    "mileage_col": mileage_col,
    "make_col": make_col,
    "model_col": model_col,
    "fuel_col": fuel_col,
    "vehicle_class_col": vehicle_class_col,
    "colour_col": colour_col,
    "postcode_col": postcode_col,
    "source_file_col": source_file_col
}

identified_columns

{'test_result_col': 'test_result',
 'test_date_col': 'test_date',
 'first_use_date_col': 'first_use_date',
 'mileage_col': 'test_mileage',
 'make_col': 'make',
 'model_col': 'model',
 'fuel_col': 'fuel_type',
 'vehicle_class_col': 'test_class_id',
 'colour_col': 'colour',
 'postcode_col': 'postcode_area',
 'source_file_col': 'source_file'}

In [6]:
# Create a list of useful columns that were found

useful_columns = [
    test_result_col,
    test_date_col,
    first_use_date_col,
    mileage_col,
    make_col,
    model_col,
    fuel_col,
    vehicle_class_col,
    colour_col,
    postcode_col,
    source_file_col
]

# Remove any columns that were not found
useful_columns = [col for col in useful_columns if col is not None]

mot_clean = mot_clean[useful_columns].copy()

mot_clean.head()

,test_result,test_date,first_use_date,test_mileage,make,model,fuel_type,test_class_id,colour,postcode_area,source_file
0,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,test_result_202401.csv
1,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,test_result_202401.csv
2,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,test_result_202401.csv
3,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,test_result_202401.csv
4,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,test_result_202401.csv


In [7]:
# Rename columns to standard project names

rename_map = {}

if test_result_col:
    rename_map[test_result_col] = "test_result"

if test_date_col:
    rename_map[test_date_col] = "test_date"

if first_use_date_col:
    rename_map[first_use_date_col] = "first_use_date"

if mileage_col:
    rename_map[mileage_col] = "test_mileage"

if make_col:
    rename_map[make_col] = "make"

if model_col:
    rename_map[model_col] = "model"

if fuel_col:
    rename_map[fuel_col] = "fuel_type"

if vehicle_class_col:
    rename_map[vehicle_class_col] = "vehicle_class"

if colour_col:
    rename_map[colour_col] = "colour"

if postcode_col:
    rename_map[postcode_col] = "postcode_area"

if source_file_col:
    rename_map[source_file_col] = "source_file"

mot_clean = mot_clean.rename(columns=rename_map)

mot_clean.columns.tolist()

['test_result',
 'test_date',
 'first_use_date',
 'test_mileage',
 'make',
 'model',
 'fuel_type',
 'vehicle_class',
 'colour',
 'postcode_area',
 'source_file']

In [8]:
# Clean text columns

text_columns = [
    "test_result",
    "make",
    "model",
    "fuel_type",
    "colour",
    "postcode_area",
    "source_file"
]

for column in text_columns:
    if column in mot_clean.columns:
        mot_clean[column] = (
            mot_clean[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )

mot_clean.head()

,test_result,test_date,first_use_date,test_mileage,make,model,fuel_type,vehicle_class,colour,postcode_area,source_file
0,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV
1,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV
2,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV
3,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV
4,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV


## Convert Date Columns

The MOT data includes date fields such as test date and first use date.

These need to be converted into proper datetime format so that vehicle age can be calculated.

In [9]:
# Convert date columns

date_columns = ["test_date", "first_use_date"]

for column in date_columns:
    if column in mot_clean.columns:
        mot_clean[column] = pd.to_datetime(mot_clean[column], errors="coerce")

mot_clean[date_columns].dtypes

test_date         datetime64[us]
first_use_date    datetime64[us]
dtype: object

In [10]:
# Clean mileage column

if "test_mileage" in mot_clean.columns:
    mot_clean["test_mileage"] = pd.to_numeric(mot_clean["test_mileage"], errors="coerce")

mot_clean["test_mileage"].describe()

count    595096.000000
mean      78965.533521
std       48902.298514
min           2.000000
25%       42322.000000
50%       71086.000000
75%      106523.250000
max      999999.000000
Name: test_mileage, dtype: float64

## Create MOT Result Flags

The test result column will be converted into simple pass/fail indicators.

These fields will be used later for MOT failure analysis and modelling.

In [11]:
# Check MOT result values

mot_clean["test_result"].value_counts(dropna=False)

test_result
P      460451
F      107756
PRS     28170
ABR      3188
ABA       435
Name: count, dtype: Int64

In [12]:
# Create MOT outcome flags

mot_clean["is_fail"] = mot_clean["test_result"].isin([
    "F",
    "FAIL",
    "FAILED"
])

mot_clean["is_pass"] = mot_clean["test_result"].isin([
    "P",
    "PASS",
    "PASSED"
])

mot_clean["mot_outcome"] = np.where(
    mot_clean["is_fail"],
    "Fail",
    np.where(mot_clean["is_pass"], "Pass", "Other")
)

mot_clean["mot_outcome"].value_counts(dropna=False)

mot_outcome
Pass     460451
Fail     107756
Other     31793
Name: count, dtype: int64

In [13]:
# Create vehicle age at test date

if "test_date" in mot_clean.columns and "first_use_date" in mot_clean.columns:
    mot_clean["vehicle_age_years"] = (
        (mot_clean["test_date"] - mot_clean["first_use_date"]).dt.days / 365.25
    )

    # Remove unrealistic negative ages by setting them to missing
    mot_clean.loc[mot_clean["vehicle_age_years"] < 0, "vehicle_age_years"] = np.nan

mot_clean["vehicle_age_years"].describe()

count    600000.000000
mean         10.907609
std           8.338797
min           0.000000
25%           6.477755
50%           9.459274
75%          14.357290
max        1813.308693
Name: vehicle_age_years, dtype: float64

In [14]:
# Create mileage bands

if "test_mileage" in mot_clean.columns:
    mot_clean["mileage_band"] = pd.cut(
        mot_clean["test_mileage"],
        bins=[-1, 30000, 60000, 100000, 150000, 200000, np.inf],
        labels=[
            "0-30k",
            "30k-60k",
            "60k-100k",
            "100k-150k",
            "150k-200k",
            "200k+"
        ]
    )

mot_clean["mileage_band"].value_counts(dropna=False)

mileage_band
60k-100k     182161
30k-60k      154928
100k-150k    122234
0-30k         86396
150k-200k     38114
200k+         11263
NaN            4904
Name: count, dtype: int64

In [15]:
# Create vehicle age bands

if "vehicle_age_years" in mot_clean.columns:
    mot_clean["vehicle_age_band"] = pd.cut(
        mot_clean["vehicle_age_years"],
        bins=[-1, 3, 5, 8, 10, 15, 20, np.inf],
        labels=[
            "0-3 years",
            "3-5 years",
            "5-8 years",
            "8-10 years",
            "10-15 years",
            "15-20 years",
            "20+ years"
        ]
    )

mot_clean["vehicle_age_band"].value_counts(dropna=False)

vehicle_age_band
5-8 years      148193
10-15 years    145260
15-20 years     91043
8-10 years      85958
3-5 years       64062
20+ years       42395
0-3 years       23089
Name: count, dtype: int64

In [16]:
# Create test month field

if "test_date" in mot_clean.columns:
    mot_clean["test_month"] = mot_clean["test_date"].dt.to_period("M").astype("string")

mot_clean["test_month"].value_counts().sort_index()

test_month
2024-01    50000
2024-02    50000
2024-03    50000
2024-04    50000
2024-05    50000
2024-06    50000
2024-07    50000
2024-08    50000
2024-09    50000
2024-10    50000
2024-11    50000
2024-12    50000
Name: count, dtype: Int64

In [17]:
# Final cleaned dataset checks

print("Rows:", mot_clean.shape[0])
print("Columns:", mot_clean.shape[1])

mot_clean.head()

Rows: 600000
Columns: 18


,test_result,test_date,first_use_date,test_mileage,make,model,fuel_type,vehicle_class,colour,postcode_area,source_file,is_fail,is_pass,mot_outcome,vehicle_age_years,mileage_band,vehicle_age_band,test_month
0,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV,False,True,Pass,13.336071,60k-100k,10-15 years,2024-01
1,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV,False,True,Pass,13.336071,60k-100k,10-15 years,2024-01
2,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV,False,True,Pass,13.336071,60k-100k,10-15 years,2024-01
3,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV,False,True,Pass,13.336071,60k-100k,10-15 years,2024-01
4,P,2024-01-02,2010-09-01,71614.0,PORSCHE,911,PE,4,RED,OX,TEST_RESULT_202401.CSV,False,True,Pass,13.336071,60k-100k,10-15 years,2024-01


In [18]:
# Missing values in cleaned dataset

missing_cleaned = (
    mot_clean
    .isna()
    .sum()
    .reset_index()
)

missing_cleaned.columns = ["column_name", "missing_values"]
missing_cleaned["missing_percentage"] = round(
    missing_cleaned["missing_values"] / len(mot_clean) * 100,
    2
)

missing_cleaned.sort_values(by="missing_values", ascending=False)

,column_name,missing_values,missing_percentage
15,mileage_band,4904,0.82
3,test_mileage,4904,0.82
5,model,1,0.00
0,test_result,0,0.00
1,test_date,0,0.00
2,first_use_date,0,0.00
6,fuel_type,0,0.00
7,vehicle_class,0,0.00
8,colour,0,0.00
4,make,0,0.00


## Export Cleaned MOT Sample Dataset

The cleaned MOT sample dataset will be saved into the processed data folder.

This file will be used for exploratory analysis, SQL reporting, Power BI dashboarding and future modelling.

In [19]:
# Export cleaned sample dataset

mot_clean.to_csv(cleaned_output_file, index=False)

print("Cleaned dataset exported successfully.")
print("Output file:", cleaned_output_file)
print("File exists:", cleaned_output_file.exists())

Cleaned dataset exported successfully.
Output file: ..\data\processed\mot_2024_cleaned_sample.csv
File exists: True


## Cleaning Summary

This notebook cleaned and prepared the 2024 MOT sample dataset.

Key preparation steps included:

- Standardising column names
- Selecting useful fields for MOT failure analysis
- Cleaning text fields
- Converting date fields
- Cleaning mileage values
- Creating pass/fail indicators
- Creating vehicle age fields
- Creating mileage bands
- Creating vehicle age bands
- Creating test month fields
- Exporting a cleaned sample dataset

The next notebook will focus on exploratory data analysis to understand MOT failure patterns by mileage, vehicle age, make, model, fuel type and vehicle class.